In [ ]:
# 1. IMPORT LIBRARIES, CONFIGURE PATHS & ENVIRONMENT SETUP
# ------------------------------------------------------------

import json
import time
import os
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# PROJECT PATH SETUP
PROJECT_ROOT = Path(r"E:\STREAMINTEL360_Complete")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

ARTIFACT_ROOT = ARTIFACTS_DIR / "notebook_07_rag"
KNOWLEDGE_DIR = ARTIFACT_ROOT / "knowledge_base"
REPORTS_DIR = ARTIFACT_ROOT / "reports"
METADATA_DIR = ARTIFACT_ROOT / "metadata"

for dir_path in [KNOWLEDGE_DIR, REPORTS_DIR, METADATA_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Loading .env and Gemini API key
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(f"GEMINI_API_KEY could not be loaded from {ENV_PATH}")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = "gemini-3.1-flash-lite"

print("STREAMINTEL 360 — RAG ENVIRONMENT INITIALIZATION")
print(f"Project       : STREAMINTEL 360")
print(f"Module        : RAG Decision Engine")
print(f"Artifacts Dir : {ARTIFACTS_DIR.resolve()}")
print(f"Artifact Root : {ARTIFACT_ROOT.resolve()}")
print(f"LLM Model     : {MODEL_NAME}")
print("Retrieval     : TF-IDF + Cosine Similarity")
print("API Key       : Successfully loaded")


STREAMINTEL 360 — RAG ENVIRONMENT INITIALIZATION
Project       : STREAMINTEL 360
Module        : RAG Decision Engine
Artifacts Dir : E:\STREAMINTEL360_Complete\artifacts
Artifact Root : E:\STREAMINTEL360_Complete\artifacts\notebook_07_rag
LLM Model     : gemini-3.1-flash-lite
Retrieval     : TF-IDF + Cosine Similarity
API Key       : Successfully loaded


In [2]:
# 2. LOAD VERIFIED ANALYTICS EVIDENCE (FROM NOTEBOOK 06)
# ------------------------------------------------------------

PAYLOAD_FILE = ARTIFACTS_DIR / "notebook_06_llm" / "metrics" / "streamintel_analytics_payload.json"

if not PAYLOAD_FILE.exists():
    raise FileNotFoundError(
        f"Analytics payload not found:\n{PAYLOAD_FILE}\n"
        "Run Notebook 06 before Notebook 07."
    )

with open(PAYLOAD_FILE, "r", encoding="utf-8") as f:
    analytics_payload = json.load(f)

print("Verified analytics payload loaded successfully.")
print(f"Source : {PAYLOAD_FILE}")
print("\nAvailable evidence modules:")
for key in analytics_payload:
    print(f"• {key}")


Verified analytics payload loaded successfully.
Source : E:\STREAMINTEL360_Complete\artifacts\notebook_06_llm\metrics\streamintel_analytics_payload.json

Available evidence modules:
• architecture_note
• platform_metadata
• subscriber_churn
• demand_forecasting
• recommendation_engine
• computer_vision
• nlp_sentiment


In [ ]:
# 3. BUILD PROJECT KNOWLEDGE BASE
# ------------------------------------------------------------
knowledge_documents = []

def add_document(doc_id: str, title: str, content: str, source: str):
    knowledge_documents.append({
        "doc_id": doc_id,
        "title": title,
        "content": content.strip(),
        "source": source,
    })


# Platform Architecture
add_document(
    "platform_architecture",
    "StreamIntel 360 Architecture",
    """
STREAMINTEL 360 is an AI-powered streaming analytics and decision
intelligence platform combining subscriber churn prediction, demand
forecasting, a hybrid recommendation engine, computer vision genre
classification, NLP sentiment analysis, LLM executive intelligence,
RAG decision support, explainable AI, and an integration API.

Notebook 06 is an inference-only LLM layer: it consumes verified
analytics from Notebooks 01-05 and generates an evidence-grounded
executive report. No model is trained in Notebook 06 or Notebook 07.
""",
    "Project Architecture",
)

# Churn 
churn = analytics_payload["subscriber_churn"]
add_document(
    "churn_module",
    "Subscriber Churn Intelligence",
    f"""
Notebook 01 — Subscriber Churn.

Task: {churn['task']}
Status: {churn['status']}
Best Model: {churn['best_model']}
Verified Metrics: {json.dumps(churn.get('verified_metrics') or {}, indent=2)}
Train Samples: {churn.get('train_samples')}
Test Samples: {churn.get('test_samples')}
Features Used: {churn.get('num_features')}
""",
    "Notebook 01",
)

# Forecasting 
forecast = analytics_payload["demand_forecasting"]
add_document(
    "forecasting_module",
    "Demand Forecasting Intelligence",
    f"""
Notebook 02 — Demand Forecasting.

Task: {forecast['task']}
Best Model (by RMSE): {forecast['best_model_by_rmse']}
Best Model Metrics: {json.dumps(forecast.get('best_model_metrics') or {}, indent=2)}
All Models Compared: {json.dumps(forecast.get('all_models_compared') or [], indent=2)}
""",
    "Notebook 02",
)

# Recommendation Engine
recommendation = analytics_payload["recommendation_engine"]
add_document(
    "recommendation_module",
    "Recommendation Intelligence",
    f"""
Notebook 03 — Hybrid Recommendation Engine.

Task: {recommendation['task']}
Status: {recommendation['status']}
Verified Evaluation Metrics: {json.dumps(recommendation.get('verified_evaluation_metrics') or {}, indent=2)}
Optimal Hybrid Weights: {json.dumps(recommendation.get('optimal_hybrid_weights') or {}, indent=2)}
""",
    "Notebook 03",
)

# Computer Vision 
vision = analytics_payload["computer_vision"]
add_document(
    "computer_vision_module",
    "Computer Vision Intelligence",
    f"""
Notebook 04 — Computer Vision.

Task: {vision['task']}
Final Model: {vision['final_model']}
Input Size: {vision['input_size']}
Decision Threshold: {vision['decision_threshold']}
Number of Genre Classes: {vision.get('num_classes')}
Verified Metrics: {json.dumps(vision.get('verified_metrics') or {}, indent=2)}
""",
    "Notebook 04",
)

# NLP 
nlp = analytics_payload["nlp_sentiment"]
add_document(
    "nlp_module",
    "NLP Sentiment Intelligence",
    f"""
Notebook 05 — NLP and Sentiment Analysis.

Task: {nlp['task']}
Dataset: {nlp['dataset']}
Model: {nlp['model']}
TF-IDF Vocabulary Size: {nlp.get('tfidf_vocabulary_size')}
Verified Metrics: {json.dumps(nlp.get('verified_metrics') or {}, indent=2)}
""",
    "Notebook 05",
)

# Save Knowledge Base 
knowledge_file = KNOWLEDGE_DIR / "rag_knowledge_base.json"
with open(knowledge_file, "w", encoding="utf-8") as f:
    json.dump(knowledge_documents, f, indent=2, ensure_ascii=False)

print("RAG knowledge base created successfully.")
print(f"Documents : {len(knowledge_documents)}")
print(f"Saved To  : {knowledge_file}")


RAG knowledge base created successfully.
Documents : 6
Saved To  : E:\STREAMINTEL360_Complete\artifacts\notebook_07_rag\knowledge_base\rag_knowledge_base.json


In [ ]:
# 4. TF-IDF RETRIEVAL INDEX
# ------------------------------------------------------------

documents_text = [f"{doc['title']}\n{doc['content']}" for doc in knowledge_documents]
vectorizer = TfidfVectorizer(lowercase=True,stop_words="english",ngram_range=(1, 2),max_features=10000,)
document_matrix = vectorizer.fit_transform(documents_text)

print("RAG retrieval index created successfully.")
print(f"Documents        : {document_matrix.shape[0]}")
print(f"Vocabulary Size  : {document_matrix.shape[1]}")
print("Retrieval Method : TF-IDF + Cosine Similarity")


RAG retrieval index created successfully.
Documents        : 6
Vocabulary Size  : 441
Retrieval Method : TF-IDF + Cosine Similarity


In [5]:
# 5. EVIDENCE RETRIEVER
# ------------------------------------------------------------

def retrieve_documents(query: str, top_k: int = 3) -> list:
    """Retrieves the most relevant project knowledge documents using TF-IDF cosine similarity."""
    if not query or not query.strip():
        raise ValueError("Query cannot be empty.")

    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, document_matrix).flatten()
    ranked_indices = similarities.argsort()[::-1][:top_k]

    results = []
    for index in ranked_indices:
        results.append({
            "doc_id": knowledge_documents[index]["doc_id"],
            "title": knowledge_documents[index]["title"],
            "source": knowledge_documents[index]["source"],
            "content": knowledge_documents[index]["content"],
            "similarity_score": round(float(similarities[index]), 4),
        })
    return results


# Retriever smoke test
test_query = "Which model performs best for demand forecasting?"
retrieved_docs = retrieve_documents(test_query, top_k=3)

print("Retriever test query:")
print(test_query)
print("\nRetrieved evidence:")
for rank, doc in enumerate(retrieved_docs, start=1):
    print(f"{rank}. {doc['title']} (score={doc['similarity_score']})")


Retriever test query:
Which model performs best for demand forecasting?

Retrieved evidence:
1. Demand Forecasting Intelligence (score=0.282)
2. StreamIntel 360 Architecture (score=0.1186)
3. Subscriber Churn Intelligence (score=0.0728)


In [ ]:
# 6. EVIDENCE-GROUNDED RAG PROMPT
# ------------------------------------------------------------

def build_rag_prompt(query: str, retrieved_docs: list) -> str:
    """Builds an evidence-grounded RAG prompt restricted to retrieved project evidence."""
    evidence_blocks = []
    for rank, doc in enumerate(retrieved_docs, start=1):
        evidence_blocks.append(f"""
[EVIDENCE {rank}]
Title: {doc['title']}
Source: {doc['source']}
Similarity Score: {doc['similarity_score']}

{doc['content']}
""")
    evidence_context = "\n".join(evidence_blocks)
    return f"""
You are the Decision Intelligence Assistant for STREAMINTEL 360.

Answer the user's question using ONLY the retrieved project
evidence provided below.

USER QUESTION:
{query}

RETRIEVED EVIDENCE:
{evidence_context}

RULES:
1. Do not invent metrics or facts.
2. Do not use information outside the supplied evidence.
3. If the evidence is insufficient, say:
   "Insufficient evidence in the retrieved project knowledge."
4. Clearly distinguish verified findings from recommendations.
5. Do not claim causation unless explicitly supported.
6. Do not present proposed integrations as existing capabilities.
7. Keep the response concise and business-oriented.
8. When useful, mention the source notebook.

Return a clear decision-oriented answer.
"""

print("RAG prompt builder ready.")

RAG prompt builder ready.


In [ ]:
# 7. GEMINI RAG GENERATION
# ------------------------------------------------------------

def generate_rag_answer(query: str, top_k: int = 3) -> dict:
    """Retrieves relevant evidence and generates an evidence-grounded answer using Gemini."""
    start_time = time.time()

    retrieved = retrieve_documents(query, top_k=top_k)
    rag_prompt = build_rag_prompt(query, retrieved)

    response = client.models.generate_content(model=MODEL_NAME, contents=rag_prompt)
    answer = getattr(response, "text", None)

    if not answer or not answer.strip():
        raise ValueError("Gemini returned an empty RAG response.")

    latency = time.time() - start_time

    return {
        "query": query,
        "answer": answer.strip(),
        "retrieved_evidence": retrieved,
        "latency_seconds": round(latency, 3),
    }

print("RAG generation pipeline ready.")

RAG generation pipeline ready.


In [ ]:
# 8. RAG DECISION INTELLIGENCE TEST
# ------------------------------------------------------------

query = (
    "Which demand forecasting model should be prioritized, "
    "and what tradeoff should decision-makers consider?"
)

rag_result = generate_rag_answer(query, top_k=3)
print("STREAMINTEL 360 — RAG DECISION INTELLIGENCE")
print("\nQUESTION")
print(rag_result["query"])

print("\nANSWER")
print(rag_result["answer"])

print("\nRETRIEVED EVIDENCE")
for rank, evidence in enumerate(rag_result["retrieved_evidence"], start=1):
    print(f"{rank}. {evidence['title']} | Source: {evidence['source']} | Score: {evidence['similarity_score']}")

print(f"\nRAG Latency: {rag_result['latency_seconds']:.3f} seconds")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


STREAMINTEL 360 — RAG DECISION INTELLIGENCE

QUESTION
Which demand forecasting model should be prioritized, and what tradeoff should decision-makers consider?

ANSWER
Based on the demand forecasting analysis provided in Notebook 02, here is the decision-oriented assessment:

**Recommendation**
The **LSTM** (Long Short-Term Memory) model should be prioritized for hourly streaming demand forecasting. It is ranked as the best-performing model based on RMSE (12,779.12), outperforming alternative models such as GRU, SimpleRNN, SARIMA, and ARIMA.

**Decision-Makers’ Tradeoff**
Decision-makers should consider the performance gap between the deep learning models (LSTM, GRU, SimpleRNN) and traditional statistical models (SARIMA, ARIMA). While the LSTM model offers superior accuracy (lowest RMSE and MAE), the evidence indicates that the traditional models (SARIMA and ARIMA) show significantly different MAPE values (0.39 and 0.51 respectively) compared to the deep learning models (ranging from 8.

In [9]:
# 9. ARTIFACT PERSISTENCE & VERIFICATION
# ------------------------------------------------------------

result_file = REPORTS_DIR / "rag_decision_result.json"
with open(result_file, "w", encoding="utf-8") as f:
    json.dump(rag_result, f, indent=2, ensure_ascii=False)

config = {
    "project": "STREAMINTEL 360",
    "module": "RAG Decision Engine",
    "retrieval_method": "TF-IDF + Cosine Similarity",
    "embedding_method": "TF-IDF",
    "vectorizer_parameters": {
        "ngram_range": [1, 2],
        "max_features": 10000,
        "stop_words": "english",
    },
    "llm_model": MODEL_NAME,
    "knowledge_documents": len(knowledge_documents),
}

config_file = METADATA_DIR / "rag_config.json"
with open(config_file, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("=" * 70)
print("STREAMINTEL 360 — NOTEBOOK 07 COMPLETE")
print("=" * 70)
print(f"Knowledge Base : {knowledge_file}")
print(f"RAG Result     : {result_file}")
print(f"Configuration  : {config_file}")
print(f"Documents      : {len(knowledge_documents)}")
print(f"LLM Model      : {MODEL_NAME}")
print("Retrieval      : TF-IDF + Cosine Similarity")
print("Status         : SUCCESS")


STREAMINTEL 360 — NOTEBOOK 07 COMPLETE
Knowledge Base : E:\STREAMINTEL360_Complete\artifacts\notebook_07_rag\knowledge_base\rag_knowledge_base.json
RAG Result     : E:\STREAMINTEL360_Complete\artifacts\notebook_07_rag\reports\rag_decision_result.json
Configuration  : E:\STREAMINTEL360_Complete\artifacts\notebook_07_rag\metadata\rag_config.json
Documents      : 6
LLM Model      : gemini-3.1-flash-lite
Retrieval      : TF-IDF + Cosine Similarity
Status         : SUCCESS
